# [실습9] Image Captioning: Transformer로 이미지 설명하기

이미지를 보고 자연어 설명을 자동 생성하는 AI를 만들어봅니다 📸

## 📚 실습 목표

1. **Image Captioning** — 이미지를 입력하면 영어 설명을 자동 생성
2. **BLIP 모델** — Vision Transformer(ViT) + Transformer Decoder의 결합
3. **한국어 번역** — MarianMT로 영어 캡션을 한국어로 자동 번역
4. **BLEU 평가** — 생성된 캡션과 정답 캡션의 유사도 자동 측정

## 🔄 처리 파이프라인

```
[입력 이미지]
    ↓
[ViT 인코더] — 이미지를 벡터로 변환 (실습5-6의 ViT와 동일!)
    ↓
[Transformer 디코더] — 벡터를 보고 영어 설명 생성 (실습8의 Decoder와 동일!)
    ↓
[영어 캡션]
    ↓
[MarianMT 번역] — 영어 → 한국어 자동 번역
    ↓
[한국어 캡션] ← MSCOCO 정답과 비교 평가!
```

## 📖 핵심 개념

| 개념 | 설명 | 이전 실습 연결 |
|------|------|----------------|
| **Image Captioning** | 이미지 → 텍스트 자동 생성 | 실습8 Encoder-Decoder 응용 |
| **BLIP** | Vision-Language 사전학습 모델 | 실습5-6 ViT/CLIP의 발전형 |
| **ViT Encoder** | 이미지를 패치 단위로 Attention | 실습5 ViT, 실습6 CLIP |
| **BLEU Score** | n-gram 기반 자동 평가 | 실습8 번역 평가 지표 |

## ⏱️ 실습 정보

| 항목 | 내용 |
|------|------|
| 예상 시간 | ~20분 (모델 다운로드 포함) |
| 난이도 | ⭐⭐☆☆☆ (HuggingFace pipeline 활용) |
| 환경 | CPU/MPS (GPU 불필요) |
| 주요 라이브러리 | transformers, torch, PIL, nltk |
| AI Hub | https://aihub.or.kr/aihubdata/data/view.do?srchOptnCnd=OPTNCND001&currMenu=115&topMenu=100&searchKeyword=ms+coco&aihubDataSe=data&dataSetSn=261 |

## 📝 과제 안내

이 노트북은 **실습 과제용**입니다. 본문 중 아래 5곳에 `# TODO` 표시와 단계별 힌트만 남기고 BLIP/MarianMT 추론 로직을 비워두었습니다.

1. Step 4 — 단일 이미지 캡셔닝 테스트 (`processor` → `generate` → `decode`)
2. Step 4 — 조건부 vs 무조건부 캡셔닝 비교 (`text=` 조건부 생성)
3. Step 4 — 50장 전체 캡셔닝 루프 (1번과 같은 패턴을 반복 적용)
4. Step 5 — BLEU 점수 계산 (`references` / `hypothesis` 만들어 `sentence_bleu` 호출)
5. Step 6 — `translate_to_korean()` 함수 (MarianMT `tokenize` → `generate` → `decode`)

모두 채워야 이어지는 셀(시각화·평균 BLEU·번역 결과 비교)이 정상 동작합니다.

맨 끝의 **"🎯 [TODO] 직접 해보기"** 섹션(자신만의 이미지로 캡셔닝 + large 모델 비교, 2개 과제)은 정답이 없는 자유 과제이니 원하는 대로 시도해보세요.

## 🗺️ 전체 학습 여정에서의 위치

```
실습1-4: NLP 기초 → Seq2Seq → Attention → Transformer 번역
                                                    ↓
실습5:   Chroma Vector DB (임베딩 검색)      ← ViT로 이미지 임베딩
실습6:   CLIP 멀티모달 검색                  ← 이미지 + 텍스트 매칭
실습7:   FastAPI 배포                        ← 실전 서비스화
실습8:   Transformer 직접 구현               ← Encoder-Decoder 핵심 이해
                                                    ↓
실습9:   Image Captioning (지금!)            ← 모든 것의 종합!
         ViT(실습5-6) + Decoder(실습8) + 번역 + 평가
```

### CLIP vs BLIP: 무엇이 다른가?

| | CLIP (실습6) | BLIP (실습9) |
|---|---|---|
| **목적** | 이미지-텍스트 **매칭** (유사도) | 이미지 → 텍스트 **생성** |
| **방식** | Contrastive Learning | Generative (Auto-regressive) |
| **출력** | 유사도 점수 | 새로운 문장 |
| **활용** | 검색, 분류 | 캡셔닝, VQA |
| **Decoder** | 없음 (인코더만) | Transformer Decoder 사용 |

## 🏗️ BLIP 아키텍처

BLIP(Bootstrapping Language-Image Pre-training)은 실습8에서 구현한 Transformer와 같은 구조입니다:

```
                    BLIP 모델
┌─────────────────────────────────────────┐
│                                         │
│  [입력 이미지]                          │
│       ↓                                 │
│  ┌──────────────────┐                   │
│  │  ViT 인코더      │ ← 실습5-6의 ViT! │
│  │  (이미지 → 벡터) │                   │
│  └──────┬───────────┘                   │
│         ↓                               │
│    이미지 특징 벡터                      │
│         ↓                               │
│  ┌──────────────────────────────┐       │
│  │  Transformer 디코더          │       │
│  │  ┌────────────────────────┐  │       │
│  │  │ Masked Self-Attention  │  │       │
│  │  │ (실습8과 동일!)        │  │       │
│  │  └────────┬───────────────┘  │       │
│  │           ↓                  │       │
│  │  ┌────────────────────────┐  │       │
│  │  │ Cross-Attention        │  │       │
│  │  │ (이미지 벡터 참조)     │  │       │
│  │  │ (실습8의 Enc-Dec Attn!)│  │       │
│  │  └────────┬───────────────┘  │       │
│  │           ↓                  │       │
│  │  ┌────────────────────────┐  │       │
│  │  │ Feed-Forward Network   │  │       │
│  │  └────────┬───────────────┘  │       │
│  └───────────┼──────────────────┘       │
│              ↓                          │
│  "a man riding a motorcycle"            │
│  (영어 캡션 자동 생성)                  │
└─────────────────────────────────────────┘
```

**실습8 복습:** Encoder가 입력을 읽고, Decoder가 Cross-Attention으로 Encoder 출력을 참조하며 토큰을 하나씩 생성!  
BLIP은 이 구조에서 Encoder를 ViT(이미지용)로, Decoder를 텍스트 생성용으로 사용합니다.

## 🛠️ Step 1: 환경 준비

In [ ]:
%pip install -q transformers torch torchvision pillow requests matplotlib pandas numpy nltk tqdm

In [ ]:
import json
import os
import random
import warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import requests
import nltk

# BLEU 평가를 위한 nltk 데이터
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# 한글 폰트 설정 (macOS)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 디바이스 설정
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 디바이스: {device}")

## 📊 Step 2: MSCOCO 한국어 캡셔닝 데이터 탐색

**MSCOCO (Microsoft Common Objects in Context)**는 컴퓨터 비전 분야의 대표적인 벤치마크 데이터셋입니다.  
AI Hub에서 제공하는 한국어 버전은 각 이미지에 **영어 + 한국어 캡션 5개씩**이 포함되어 있습니다.

In [ ]:
# MSCOCO 한국어 캡셔닝 데이터 로드
data_path = "korean_image_captioning_dataset/MSCOCO_train_val_Korean.json"
with open(data_path, 'r', encoding='utf-8') as f:
    coco_data = json.load(f)

print(f"전체 데이터 수: {len(coco_data):,}개")

# train/val 분류
val_data = [d for d in coco_data if "val2014" in d["file_path"]]
train_data = [d for d in coco_data if "train2014" in d["file_path"]]
print(f"  - train2014: {len(train_data):,}개")
print(f"  - val2014:   {len(val_data):,}개")

# 샘플 확인
print("\n=== 데이터 예시 ===")
sample = val_data[0]
print(json.dumps(sample, ensure_ascii=False, indent=2))

In [ ]:
# 캡션 다양성 확인: 같은 이미지, 5가지 다른 설명
print("=== 같은 이미지에 대한 5가지 캡션 ===\n")
sample = val_data[0]
print(f"이미지: {sample['file_path']}\n")

print("영어 캡션:")
for i, cap in enumerate(sample['captions'], 1):
    print(f"  {i}. {cap}")

print("\n한국어 캡션:")
for i, cap in enumerate(sample['caption_ko'], 1):
    print(f"  {i}. {cap}")

In [ ]:
# val2014에서 50장 샘플링 (재현성을 위해 seed 고정)
random.seed(42)
sample_data = random.sample(val_data, 50)
print(f"샘플링: val2014에서 {len(sample_data)}장 선택")

# 캡션 길이 분포 비교
en_lengths = [len(cap.split()) for d in sample_data for cap in d['captions']]
ko_lengths = [len(cap) for d in sample_data for cap in d['caption_ko']]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(en_lengths, bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('영어 캡션 길이 (단어 수)')
axes[0].set_xlabel('단어 수')
axes[0].set_ylabel('빈도')

axes[1].hist(ko_lengths, bins=20, color='coral', edgecolor='white')
axes[1].set_title('한국어 캡션 길이 (글자 수)')
axes[1].set_xlabel('글자 수')
axes[1].set_ylabel('빈도')

plt.tight_layout()
plt.show()

## 🖼️ Step 3: COCO 이미지 다운로드

COCO 이미지는 개별 URL로 접근할 수 있습니다.  
전체 val2014(~6GB)를 받을 필요 없이, **50장만 선택적으로 다운로드**합니다.

```
URL 패턴: http://images.cocodataset.org/val2014/COCO_val2014_000000{id}.jpg
```

In [ ]:
# COCO 이미지 50장 다운로드
IMAGE_DIR = "coco_images"
os.makedirs(IMAGE_DIR, exist_ok=True)

base_url = "http://images.cocodataset.org/"
success_count = 0

for item in tqdm(sample_data, desc="이미지 다운로드 중"):
    filename = os.path.basename(item["file_path"])
    save_path = os.path.join(IMAGE_DIR, filename)
    
    if os.path.exists(save_path):
        success_count += 1
        continue
    
    try:
        url = base_url + item["file_path"]
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        with open(save_path, "wb") as f:
            f.write(response.content)
        success_count += 1
    except Exception as e:
        print(f"  실패: {filename} - {e}")

print(f"\n다운로드 완료: {success_count}/{len(sample_data)}장")

In [ ]:
# 다운로드된 이미지 미리보기 (6장)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, item in enumerate(sample_data[:6]):
    filename = os.path.basename(item["file_path"])
    img_path = os.path.join(IMAGE_DIR, filename)
    img = Image.open(img_path)
    
    axes[idx].imshow(img)
    ko_cap = item['caption_ko'][0]
    title = ko_cap[:30] + "..." if len(ko_cap) > 30 else ko_cap
    axes[idx].set_title(title, fontsize=10)
    axes[idx].axis('off')

plt.suptitle("MSCOCO 이미지 + 한국어 캡션", fontsize=14)
plt.tight_layout()
plt.show()

## 🤖 Step 4: BLIP 모델로 Image Captioning

**BLIP (Salesforce/blip-image-captioning-base)** 모델을 로드합니다.

- `BlipProcessor`: 이미지 전처리 (리사이즈, 정규화) + 텍스트 토큰화
- `BlipForConditionalGeneration`: ViT Encoder + Transformer Decoder

실습8에서 `Encoder → Decoder → 토큰 생성`의 추론 과정을 구현했는데,  
BLIP도 정확히 같은 과정으로 이미지 캡션을 생성합니다!

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

print("BLIP 모델 로드 중... (최초 실행 시 ~990MB 다운로드)")
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = blip_model.to(device)
blip_model.eval()

# 모델 파라미터 수
total_params = sum(p.numel() for p in blip_model.parameters())
print(f"모델 로드 완료! (파라미터: {total_params:,}개)")

In [ ]:
# 단일 이미지 캡셔닝 테스트
item = sample_data[0]
filename = os.path.basename(item["file_path"])
img = Image.open(os.path.join(IMAGE_DIR, filename)).convert("RGB")

# TODO: 아래 순서대로 구현하세요
# 1. processor(img, return_tensors="pt")로 이미지를 전처리하고 .to(device)로 옮기세요 (변수명: inputs)
# 2. torch.no_grad() 안에서 blip_model.generate(**inputs, max_new_tokens=50)로 캡션 토큰을 생성하세요 (변수명: output)
# 3. processor.decode(output[0], skip_special_tokens=True)로 텍스트로 변환하세요 (변수명: generated_caption)
pass

# 결과 표시
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.imshow(img)
ax.axis('off')
plt.show()

print(f"🤖 BLIP 생성: {generated_caption}")
print(f"\n📝 정답 캡션 (영어):")
for i, cap in enumerate(item['captions'], 1):
    print(f"   {i}. {cap}")
print(f"\n📝 정답 캡션 (한국어):")
for i, cap in enumerate(item['caption_ko'], 1):
    print(f"   {i}. {cap}")

In [ ]:
# 조건부 vs 무조건부 캡셔닝 비교
item = sample_data[1]
filename = os.path.basename(item["file_path"])
img = Image.open(os.path.join(IMAGE_DIR, filename)).convert("RGB")

# TODO: 아래 순서대로 구현하세요
# 1. 무조건부: processor(img, return_tensors="pt").to(device) → inputs_free
#    torch.no_grad() 안에서 blip_model.generate(**inputs_free, max_new_tokens=50) → out1
#    processor.decode(out1[0], skip_special_tokens=True) → caption_free
# 2. 조건부: processor(img, text="a photo of", return_tensors="pt").to(device) → inputs_guided
#    (processor에 text= 인자를 추가하면 그 텍스트로 시작하도록 캡션 생성을 유도합니다)
#    torch.no_grad() 안에서 blip_model.generate(**inputs_guided, max_new_tokens=50) → out2
#    processor.decode(out2[0], skip_special_tokens=True) → caption_guided
pass

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.imshow(img)
ax.axis('off')
plt.show()

print(f"🔓 무조건부 생성: {caption_free}")
print(f"🔒 조건부 생성:   {caption_guided}")
print(f"\n💡 조건부 생성은 'a photo of'로 시작하도록 유도했습니다.")
print(f"   실습8의 Decoder에서 시작 토큰(BOS)이 생성 방향을 결정하는 것과 같은 원리!")

In [ ]:
# 50장 전체 캡셔닝
print("50장 이미지 캡셔닝 중...\n")
results = []

for item in tqdm(sample_data):
    filename = os.path.basename(item["file_path"])
    img_path = os.path.join(IMAGE_DIR, filename)
    
    if not os.path.exists(img_path):
        continue
    
    img = Image.open(img_path).convert("RGB")
    
    # TODO: 아래 순서대로 구현하세요 (Step 4에서 만든 것과 같은 패턴입니다)
    # 1. processor(img, return_tensors="pt").to(device) → inputs
    # 2. torch.no_grad() 안에서 blip_model.generate(**inputs, max_new_tokens=50) → output
    # 3. processor.decode(output[0], skip_special_tokens=True) → generated
    pass
    
    results.append({
        "id": item["id"],
        "file_path": item["file_path"],
        "generated_en": generated,
        "ground_truth_en": item["captions"],
        "ground_truth_ko": item["caption_ko"]
    })

print(f"\n캡셔닝 완료: {len(results)}장")

# 결과 미리보기
print("\n=== 처음 5개 결과 ===")
for r in results[:5]:
    print(f"\n🤖 생성: {r['generated_en']}")
    print(f"📝 정답: {r['ground_truth_en'][0]}")

## 📏 Step 5: BLEU 점수로 평가

**BLEU (Bilingual Evaluation Understudy)**는 생성된 텍스트와 정답 텍스트의 n-gram 유사도를 측정합니다.

실습8에서 번역 모델의 성능을 BLEU로 평가했는데, 캡셔닝에서도 동일하게 사용합니다!

```
BLEU 점수 계산:
  - 생성된 캡션의 n-gram이 정답 캡션에 얼마나 등장하는지 측정
  - 5개의 정답 캡션을 모두 참조 (reference)로 사용
  - 0.0 ~ 1.0 범위 (높을수록 좋음)
```

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smoother = SmoothingFunction().method1
bleu_scores = []

for r in results:
    # TODO: 아래 순서대로 구현하세요
    # 1. r["ground_truth_en"]의 각 캡션을 소문자로 바꾸고 단어 단위로 split()해 참조 리스트를 만드세요 → references
    #    (references는 "리스트의 리스트" 형태여야 합니다 — 정답 캡션 5개 = 참조 5개)
    # 2. r["generated_en"]을 소문자로 바꾸고 split()해 → hypothesis
    # 3. sentence_bleu(references, hypothesis, smoothing_function=smoother)로 점수를 계산하세요 → score
    # 4. bleu_scores 리스트에 score를 추가하고, r["bleu"]에도 저장하세요
    pass

avg_bleu = np.mean(bleu_scores)
print(f"평균 BLEU 점수: {avg_bleu:.4f}")
print(f"최고: {max(bleu_scores):.4f}")
print(f"최저: {min(bleu_scores):.4f}")

# BLEU 점수 분포
plt.figure(figsize=(10, 4))
plt.hist(bleu_scores, bins=20, color='steelblue', edgecolor='white')
plt.axvline(avg_bleu, color='red', linestyle='--', label=f'평균: {avg_bleu:.4f}')
plt.xlabel('BLEU Score')
plt.ylabel('이미지 수')
plt.title('BLEU 점수 분포')
plt.legend()
plt.show()

In [ ]:
# 최고/최저 BLEU 이미지 시각화
sorted_results = sorted(results, key=lambda x: x["bleu"], reverse=True)

fig, axes = plt.subplots(2, 3, figsize=(16, 11))

# 상위 3개
for idx, r in enumerate(sorted_results[:3]):
    filename = os.path.basename(r["file_path"])
    img = Image.open(os.path.join(IMAGE_DIR, filename))
    axes[0][idx].imshow(img)
    axes[0][idx].set_title(f"✅ BLEU: {r['bleu']:.3f}", fontsize=11, color='green')
    axes[0][idx].axis('off')

# 하위 3개
for idx, r in enumerate(sorted_results[-3:]):
    filename = os.path.basename(r["file_path"])
    img = Image.open(os.path.join(IMAGE_DIR, filename))
    axes[1][idx].imshow(img)
    axes[1][idx].set_title(f"❌ BLEU: {r['bleu']:.3f}", fontsize=11, color='red')
    axes[1][idx].axis('off')

plt.suptitle("BLEU 점수 상위 3개 (위) vs 하위 3개 (아래)", fontsize=14)
plt.tight_layout()
plt.show()

print("=== BLEU 최고 ===")
best = sorted_results[0]
print(f"🤖 생성: {best['generated_en']}")
print(f"📝 정답: {best['ground_truth_en'][0]}")

print(f"\n=== BLEU 최저 ===")
worst = sorted_results[-1]
print(f"🤖 생성: {worst['generated_en']}")
print(f"📝 정답: {worst['ground_truth_en'][0]}")

## 🇰🇷 Step 6: 한국어 캡셔닝

BLIP은 영어 캡션만 생성하므로, **MarianMT**로 한국어로 번역합니다.  
그리고 MSCOCO 한국어 정답 캡션과 비교합니다.

```
[BLIP 영어 캡션] → [MarianMT 영→한 번역] → [한국어 캡션]
                                                  ↓
                                    MSCOCO 한국어 정답과 비교!
```

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

print("번역 모델 로드 중... (최초 실행 시 ~900MB 다운로드)")
mt_model_name = "Helsinki-NLP/opus-mt-tc-big-en-ko"
mt_tokenizer = MarianTokenizer.from_pretrained(mt_model_name)
mt_model = MarianMTModel.from_pretrained(mt_model_name)
mt_model.eval()
print("번역 모델 로드 완료!")


def translate_to_korean(text):
    """영어 텍스트를 한국어로 번역합니다."""
    # TODO: 아래 순서대로 구현하세요
    # 1. mt_tokenizer(text, return_tensors="pt", padding=True, truncation=True)로 토큰화하세요 → inputs
    # 2. torch.no_grad() 안에서 mt_model.generate(**inputs)로 번역 토큰을 생성하세요 → translated
    # 3. mt_tokenizer.decode(translated[0], skip_special_tokens=True)로 반환하세요
    pass

In [ ]:
# 생성된 영어 캡션을 한국어로 번역
print("한국어 번역 중...\n")

for r in tqdm(results):
    r["generated_ko"] = translate_to_korean(r["generated_en"])

# 결과 비교 (5개 샘플)
for r in results[:5]:
    filename = os.path.basename(r["file_path"])
    print(f"📷 {filename}")
    print(f"  🤖 영어 생성:   {r['generated_en']}")
    print(f"  🇰🇷 한국어 번역: {r['generated_ko']}")
    print(f"  📝 한국어 정답:  {r['ground_truth_ko'][0]}")
    print()

In [ ]:
# 이미지 + 캡션 비교 시각화 (6장)
fig, axes = plt.subplots(2, 3, figsize=(16, 11))
axes = axes.flatten()

for idx in range(6):
    r = results[idx]
    filename = os.path.basename(r["file_path"])
    img = Image.open(os.path.join(IMAGE_DIR, filename))
    
    axes[idx].imshow(img)
    title = r["generated_ko"][:35]
    if len(r["generated_ko"]) > 35:
        title += "..."
    axes[idx].set_title(f"🤖 {title}", fontsize=10)
    axes[idx].axis('off')

plt.suptitle("BLIP + MarianMT: 자동 한국어 캡셔닝 결과", fontsize=14)
plt.tight_layout()
plt.show()

# 전체 결과를 DataFrame으로 정리
df = pd.DataFrame([{
    "이미지": os.path.basename(r["file_path"]),
    "영어 생성": r["generated_en"],
    "한국어 번역": r["generated_ko"],
    "한국어 정답": r["ground_truth_ko"][0],
    "BLEU": f"{r['bleu']:.3f}"
} for r in results])

print("\n=== 전체 결과 (처음 10개) ===")
df.head(10)

## 🎯 [TODO] 직접 해보기

### 과제 1: 내 사진으로 캡셔닝
자신의 사진으로 캡셔닝을 시도해보세요!

💡 힌트: `Image.open("my_photo.jpg")`으로 이미지를 로드하면 됩니다.

In [ ]:
# [TODO] 자신만의 이미지로 캡셔닝을 시도해보세요!
# 아래 주석을 해제하고 이미지 경로를 수정하세요

# my_image = Image.open("my_photo.jpg").convert("RGB")
# inputs = processor(my_image, return_tensors="pt").to(device)
# with torch.no_grad():
#     output = blip_model.generate(**inputs, max_new_tokens=50)
# my_caption = processor.decode(output[0], skip_special_tokens=True)
# print(f"영어: {my_caption}")
# print(f"한국어: {translate_to_korean(my_caption)}")

### 과제 2: large 모델과 비교

`blip-image-captioning-large` 모델로 바꿔서 품질 차이를 확인해보세요!

💡 힌트: `"Salesforce/blip-image-captioning-large"` (~1.8GB)

## 📋 실습 정리

### 오늘 배운 것

| 주제 | 핵심 내용 | 한 줄 요약 |
|------|-----------|------------|
| **BLIP 모델** | ViT Encoder + Transformer Decoder | 이미지를 이해하고 텍스트를 생성하는 사전학습 모델 |
| **Image Captioning** | 이미지 → 영어 설명 자동 생성 | Encoder-Decoder 구조의 Vision-Language 응용 |
| **BLEU 평가** | 생성 캡션 vs 정답 캡션 비교 | n-gram 기반 자동 평가 (실습8과 동일 지표) |
| **한국어 캡셔닝** | MarianMT로 번역 파이프라인 구성 | 영어 모델 + 번역 모델의 조합으로 다국어 확장 |

### ✅ 체크리스트

- [ ] MSCOCO 한국어 데이터 구조를 이해했다
- [ ] BLIP의 ViT Encoder + Transformer Decoder 구조를 이해했다
- [ ] 실습8의 Encoder-Decoder와 BLIP의 연관성을 이해했다
- [ ] 조건부/무조건부 캡셔닝의 차이를 이해했다
- [ ] BLEU 점수로 캡셔닝 품질을 평가할 수 있다
- [ ] MarianMT로 영→한 번역 파이프라인을 구성할 수 있다

### 🗺️ 전체 여정 돌아보기

```
실습1: 한국어 텍스트 분석       → NLP의 시작
실습2: Seq2Seq 번역             → 시퀀스 모델의 기초
실습3: Attention 메커니즘        → Transformer의 핵심 혁신
실습4: Transformer 번역 시스템   → 완전한 번역 파이프라인
실습5: Vector DB & ViT          → 임베딩과 검색
실습6: CLIP 멀티모달 검색       → 이미지 + 텍스트 연결
실습7: FastAPI 배포             → AI를 서비스로
실습8: Transformer 직접 구현    → 핵심 원리 완전 이해
실습9: Image Captioning (지금!) → Vision + Language 통합 🎉
```

### 💡 더 알아보기

- **BLIP-2**: BLIP의 후속 모델, Q-Former로 더 효율적인 비전-언어 연결
- **Fine-tuning**: MSCOCO 한국어 데이터로 모델을 직접 학습시키기
- **Visual QA**: 이미지에 대한 질문에 답변하는 모델 (BLIP으로 가능!)
- **Video Captioning**: 동영상 설명 자동 생성으로 확장

## ❓ FAQ

**Q1: BLIP과 CLIP의 차이가 뭔가요?**
> CLIP은 이미지-텍스트 **유사도 매칭** (검색에 적합), BLIP은 이미지에서 텍스트를 **생성** (캡셔닝에 적합)

**Q2: GPU 없이도 되나요?**
> blip-image-captioning-base는 CPU에서 이미지당 1-3초로 충분히 실행 가능합니다.

**Q3: 한국어를 직접 생성하는 모델은 없나요?**
> 다국어 캡셔닝 모델(mBLIP 등)이 연구되고 있지만, 아직 영어 모델 + 번역이 더 높은 품질을 보입니다.

**Q4: 실무에서는 어떻게 쓰이나요?**
> 접근성 서비스(시각장애인 이미지 설명), SNS 자동 태깅, 이커머스 상품 설명 자동 생성 등에 활용됩니다.